# Smart MCQ Solver

## Submission By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

In [1]:
import os
import random
import sys
from pathlib import Path

import pandas as pd
import torch

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IS_KAGGLE:
    print("Running in Kaggle environment.")
    TEST_CSV_PATH = Path(
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
    )
    SUBMISSION_OUT_PATH = Path("submission.csv")

    # Resolve Model Registry
    base_model_dir = Path("/kaggle/input/smart-mcq-solver/pyTorch/local")
    if base_model_dir.exists():
        version_dirs = list(base_model_dir.glob("*"))
        MODEL_DIR = version_dirs[0] if version_dirs else base_model_dir
    else:
        MODEL_DIR = Path("/kaggle/input/smart-mcq-solver/pyTorch/local/1")

    SRC_PATH = MODEL_DIR / "src"
else:
    print("Running in local environment.")
    if Path(os.getcwd()).name == "notebooks":
        os.chdir("..")

    print(f"Current working directory: {os.getcwd()}")

    TEST_CSV_PATH = Path("dataset/test.csv")
    MODEL_DIR = Path("outputs/local")
    SRC_PATH = Path("src")
    SUBMISSION_OUT_PATH = MODEL_DIR / "submission_local_model.csv"

# add source path to python search path
if SRC_PATH.exists():
    if str(SRC_PATH.resolve()) not in sys.path:
        sys.path.append(str(SRC_PATH.resolve()))
    print(f"Added source path to sys.path: {SRC_PATH.resolve()}")
else:
    print(
        f"WARNING: Source path '{SRC_PATH}' not found. Make sure packages are importable."  # noqa: E501
    )

print(f"Test CSV path: {TEST_CSV_PATH}")
print(f"Model directory: {MODEL_DIR}")
print(f"Submission output path: {SUBMISSION_OUT_PATH}")

Running in local environment.
Current working directory: c:\PythonicVarun\GenAI_Project-May_2026
Added source path to sys.path: C:\PythonicVarun\GenAI_Project-May_2026\src
Test CSV path: dataset\test.csv
Model directory: outputs\local
Submission output path: outputs\local\submission_local_model.csv


In [2]:
from local import config

# override config directories
config.OUTPUT_DIR = MODEL_DIR
config.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device in use: {config.DEVICE}")

Device in use: cuda


In [3]:
from local.inference import load_artifacts, predict

# Check test file
if not TEST_CSV_PATH.exists():
    print("Required test dataset not found, exiting!")
    exit(1)

test_df = pd.read_csv(TEST_CSV_PATH)
print(f"Loaded test dataset with {len(test_df)} records.")

Loaded test dataset with 500 records.


In [4]:
# Load model, vocab, and retriever
model, vocab, retriever = load_artifacts()

model

BiLSTMScorer(
  (embed): Embedding(2976, 128, padding_idx=0)
  (embed_drop): Dropout(p=0.35, inplace=False)
  (bilstm): LSTM(128, 192, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  (attn): SelfAttention(
    (proj): Linear(in_features=384, out_features=384, bias=True)
    (v): Linear(in_features=384, out_features=1, bias=False)
  )
  (head): Sequential(
    (0): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Linear(in_features=384, out_features=192, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.35, inplace=False)
    (4): Linear(in_features=192, out_features=1, bias=True)
  )
)

In [5]:
# Run prediction pipeline
sub_df = predict(test_df, model, retriever, vocab, is_train=False)

# Ensure directory exists for output if running locally
SUBMISSION_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

sub_df.to_csv(SUBMISSION_OUT_PATH, index=False)
print(f"Saved submission to: {SUBMISSION_OUT_PATH.resolve()}")

Building validation dataset: 100%|██████████| 500/500 [00:03<00:00, 133.46it/s]


Saved submission to: C:\PythonicVarun\GenAI_Project-May_2026\outputs\local\submission_local_model.csv


In [6]:
sub_df.head(10)

,id,Prediction
0,1,A E D
1,2,B C D
2,3,B E C
3,4,E C A
4,5,D B C
5,6,D B A
6,7,E C A
7,8,B E C
8,9,E C D
9,10,B C E


In [7]:
# Sample Visualization
random.seed(42)

sample_indices = random.sample(range(len(test_df)), min(3, len(test_df)))

for idx in sample_indices:
    row = test_df.iloc[idx]
    pred = sub_df.iloc[idx]["Prediction"]

    print("-" * 120)
    print(f"[ID {row['id']}] PROMPT:")
    print(row["prompt"])
    print("-" * 60)
    for c in ["A", "B", "C", "D", "E"]:
        print(f"({c}) {row[c]}")
    print("-" * 60)

    ctx = retriever.retrieve(row["prompt"], top_k=1)
    print(f"Retrieved Context (Top-1): {ctx[:300]}...")
    print("-" * 60)
    print(f"Prediction (MAP@3 Order): {pred}")
    print("-" * 120)
    print()

------------------------------------------------------------------------------------------------------------------------
[ID 328] PROMPT:
Determine the correct option: What is the Erlangen program? carefully.
------------------------------------------------------------
(A) The Erlangen program is a approach of characterizing geometries based on statistics and probability, published by Felix Klein in 1872 as Vergleichende Betrachtungen über neuere geometrische Forschungen.
(B) The Erlangen program is a approach of characterizing geometries based on group concept and projective geometry, published by Felix Klein in 1872 as Vergleichende Betrachtungen über neuere geometrische Forschungen.
(C) The Erlangen program is a approach of characterizing geometries based on algebra and trigonometry, published by Felix Klein in 1872 as Vergleichende Betrachtungen über neuere geometrische Forschungen.
(D) The Erlangen program is a approach of characterizing geometries based on geometry and topology, 